# Descripción de Boston dataset

https://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html

In [3]:
# Tratamiento dedatos
import pandas as pd
import numpy as np

#Graficos
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

#Procesado y modelado
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import statsmodels as sms
#from sklearn.datasets import load_boston

from sklearn.datasets import fetch_california_housing
from sklearn.datasets import fetch_openml
from statsmodels.stats.outliers_influence import variance_inflation_factor # para calcular el VIF
# configuración de matplotlib
plt.rcParams['image.cmap']="bwr"
plt.rcParams['figure.dpi']="100"
plt.rcParams['savefig.bbox']="tight"
style.use('ggplot') or plt.style.use('ggplot')

# configuración de warnings
import warnings
warnings.filterwarnings('ignore')
# ===========================================================


In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

In [7]:
boston = pd.read_csv("Boston.csv")
boston = pd.DataFrame(boston)
boston.head()

,Unnamed: 0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2


In [ ]:
#Xorig = boston.drop(["Unnamed: 0","medv"], axis=1)
boston = boston.drop("Unnamed: 0", axis=1)
boston.head()

## Análisis Exploratorio descriptivo

In [ ]:
boston.isna().sum()  # Verificando na's


In [ ]:
boston.describe()

In [ ]:
boston.corr()

In [ ]:
sns.heatmap(boston.corr(),annot=True)

In [ ]:
sns.pairplot(boston)


### Ajuste del modelo

In [ ]:
X_orig = boston.drop("medv",axis=1)
X_orig.head()
#X_orig = pd.DataFrame(data=boston, columns= boston.drop("medv",axis=1))
#X_orig.head()




In [ ]:
y_orig = boston["medv"]
y_orig.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_orig, y_orig, random_state=1) #75% para entrenamiento, 25% para prueba

df = pd.concat([X_train,y_train], axis=1)
X_train.shape
X_test.shape
y_train.shape
y_test.shape

In [ ]:
X_constant = sm.add_constant(X_train, prepend=True)
modmult = sm.OLS(y_train, X_constant)
modmult = modmult.fit()
print(modmult.summary())

* $R^2_{adj}:0.704$
* $p_value= 1.25e-91<0.05$, entonces rechazamos $H_0$
* Pruebas individuales: Removemos las variables age, indus
* Durbin_Watson= 1.862, es decir,
* Prob(JB)=1.02e-141

## Supuestos del modelo

In [ ]:
# supocisiones del modelos LINE

resid_val =modmult.resid
fitted_val = modmult.predict()

modmult.resid.mean()

### Prueba de normalidad

$$H_0: \text{Los residuos son normales} \quad vs. \quad H_1\text{:Los los residuos NO- son normales}$$

In [ ]:
### Prueba de normalidad
sns.boxplot(y=resid_val)
sm.qqplot(resid_val, line='s')


In [ ]:
plt.hist(resid_val)



In [ ]:
stats.shapiro(modmult.resid) #from scipy import stats

In [ ]:
# Linearity in Model

sns.regplot(x=fitted_val,y = y_train, color='blue', lowess=True, line_kws={'color':'red'})
plt.title('Fitted vs Observed')

### ¿Hay linealidad en el modelo?

### conclusión de Homocedasticidad

In [ ]:
# Es mejor usar los residuales estandarizados
resid_stand= modmult.get_influence().resid_studentized_internal
sns.regplot(x=fitted_val,y = resid_stand, lowess=True, color= "blue",line_kws={'color':'red'})
plt.title('Fitted vs Residuals stand')

###  Breusch-Pagan test for equal variance (homocedasticity)

$H_0$: Igualdad de varianza en los residuales

$vs.$

$H_1$: Los residuales no tienen la igualdad de varianza

In [ ]:
bp_test = sms.stats.diagnostic.het_breuschpagan(modmult.resid, X_constant)
print(bp_test)

In [ ]:
print('\n Breush-Pagan Test: P-value=',bp_test[1])

P-value= 1.3321294922020338e-066<0.05 no hay igualdad de varianzas

## Establece la conclusión del análisis del modelo
* Prueba Global

* Pruebas individuales

* R cuadrada adj

* Análisis de residuales

# Modelo sin outliers

In [ ]:
Ls = y_train.mean()+3*y_train.std()
Li = y_train.mean()-3*y_train.std()

print('El valor máximo permitido es:',Ls)
print('El valor mínimo permitido es:',Li) # Analice el valor del precio mínimo de una casa
X_train[(y_train>Ls) | (y_train<Li)]

Puesto que el valor mínimo permitido es negativo, no representa correctamente la naturaleza del problema, que es el valor de una casa. Por tal motivo corregimos estos valores permitidos.


In [ ]:
p25 = y_train.quantile(0.25)# Q1
p75 = y_train.quantile(0.75) # Q3
iqr = p75-p25
upper =p75+1.5*iqr
lower = p25-1.5*iqr
print(upper)
print(lower)
X_train[(y_train>upper) | (y_train<lower)]

Note que el valor mínimo en el precio de una casa es positivo.

In [ ]:
### Contruimos un nuevo data set con el filtro que excluye outliers
X2_train = X_train[(y_train<=upper) & (y_train>=lower)]
y2_train = y_train[(y_train<=upper) & (y_train>=lower)]

print("New shape: ", X2_train.shape)

print("Y_train: ", y2_train.shape)
X_train.shape

## VIF
The Variance Inflation Factor is the measure of multicollinearity that exists in the set of variables that are involved in multiple regressions. Generally, the vif value above 10 indicates that there is a high correlation with the other independent variables.

[https://en.wikipedia.org/wiki/Variance_inflation_factor#See_also](https://en.wikipedia.org/wiki/Variance_inflation_factor#See_also)

Ahora analizamos el VIF del data train original

In [ ]:
X2_train.head()


In [ ]:
vif = [variance_inflation_factor(X_constant.values, i) for i in range(X_train.shape[1])]
print(pd.DataFrame({'vif': vif[0:]}, index=X2_train.columns).T)


# Analisis del VIF

Eliminemos a TAX y RAD debido a que están altamente correlacionadas

# Tarea Generar un modelo removiendo solo la variable tax

## Construcción del segundo modelo


In [ ]:
# Nuevo modelo sin la columna 'TAX'
X2 = X2_train.drop(['tax','rad'],axis=1)
X2.head()
X2_constant= sm.add_constant(X2) # import statsmodels.api as sm
modelo2= sm.OLS(y2_train,X2_constant).fit()
print(modelo2.summary())

# * Analiza la informcación del modelo
# * Verica los supuestos de los residuales

Verifica los supuestos del modelo

In [ ]:
print(modmult.summary())

In [ ]:
X3 = boston[['chas','crim','dis','lstat','nox','ptratio','rad','rm','tax','zn']]
Y = boston['medv']

X3 = sm.add_constant(X3)

mod3 = sm.OLS(Y, X3).fit()

print(mod3.summary())

In [ ]:
print(mod3.summary())

In [ ]:
resid_val = mod3.resid

sns.boxplot(y=resid_val)
plt.show()

sm.qqplot(resid_val, line='s')
plt.show()

In [ ]:
X3 = X_orig.drop(["indus","age"], axis=1)

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y_orig, random_state=1
)

X_train3 = sm.add_constant(X_train3)

mod3 = sm.OLS(y_train3, X_train3).fit()

print(mod3.summary())

## Construcción del tercer modelo modelo sin indus y age
## * Analiza la informcación del modelo
# el modelo 3 tiene un r2 ajustado de 0.706 y un aic de 2282 la mayoría de las variables son significativas y el modelo mantiene un buen ajuste aun eliminando indus y age y el otro le faltan cosas que el 3 tiene
## * Verica los supuestos de los residuales
# los residuales se distribuyen alrededor de cero y no presentan desviaciones extremas

In [ ]:
resid_val = mod3.resid

sns.boxplot(y=resid_val)
plt.show()

sm.qqplot(resid_val, line='s')
plt.show()

los residuales se distribuyen alrededor de cero y no presentan desviaciones extremas

## Con el análisis de los modelos 2 y 3, indica ¿Cuál es el mejor modelo?

In [ ]:
# el mejor modelo es el modelo 3 porque tiene un r2 ajustado ligeramente mayor 0.706 y un aic menor 2282 que el modelo 2 por lo que presenta un mejor ajuste a los datos